In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from rich import print as rich_print
# from IPython.display import HTML, display

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.basis.sectors import sector_mask_from_build_result
from qlinks.caging import (
    CageClassificationConfig,
    CageSearchConfig,
    CageSearcher,
    classify_cage_state,
    build_type1_cage_lindblad_construction,
)
from qlinks.models import (
    SquareQLMModel,
    SquareQDMModel,
    TriangularQLMModel,
    TriangularQDMModel,
    HoneycombQLMModel,
    HoneycombQDMModel,
)
from qlinks.open_system import (
    LindbladEvolutionOptions,
    McwfOptions,
    analyze_lindblad_evolution,
    initial_density_matrix,
    random_pure_state,
    pure_density_matrix,
    run_quantum_jump_trajectory,
    sample_lindblad_mcwf,
    diagnose_monitor_kernel_closure,
)
from qlinks.visualizer import (
    HamiltonianGraphStyle,
    LiouvillianGraphVisualizer,
    StochasticSchrodingerGraphVisualizer,
)

## Model Definition

In [ ]:
model = TriangularQDMModel(
    lx=4,
    ly=4,
    boundary_condition="periodic",
    winding_a=0,
    winding_b=0,
    coup_kin=-1.0,
    coup_pot=0.7,
)

In [ ]:
build_result = model.build(
    basis_solver="dfs",
    builder="bitmask",
    backend="scipy",
    sort_basis=True,
    on_missing="raise",
)

hamiltonian_matrix = build_result.hamiltonian
kinetic_matrix = build_result.kinetic
potential_matrix = build_result.potential
basis = build_result.basis
basis_configs = basis_configs_from_build_result(build_result)
sector_mask = sector_mask_from_build_result(build_result)

print("n_states =", basis.n_states)
print("K is bipartite =", nx.is_bipartite(nx.from_scipy_sparse_array(kinetic_matrix, edge_attribute="weight")))

## Search for caged states

In [ ]:
searcher = CageSearcher.from_model_build_result(
    build_result,
    config=CageSearchConfig(
        search_type="type1",
        tolerance=1e-10,
        degenerate_basis_strategy="ipr",
        ipr_n_restarts=256,
        ipr_candidate_count=128,
        ipr_random_seed=1234,
        store_full_states=True,
    ),
)

search_result = searcher.run()
print("number of cages:", len(search_result))
print("signatures:", search_result.signatures)
print("counts by signature:", search_result.counts_by_signature)

In [ ]:
signature = (0, 4)
record_index = 0
record = search_result[signature, record_index]

state_vector = record.full_state
if state_vector is None:
    state_vector = search_result.full_state_matrix()[record_index]

print("selected signature:", record.signature)
print("support size:", record.support.size)
print("energy:", record.cage_state.energy)

## Classify the selected caged state

In [ ]:
report = classify_cage_state(
    record.cage_state,
    kinetic_matrix=build_result.kinetic,
    basis_configs=basis_configs,
    sector_mask=sector_mask,
    hilbert_size=search_result.hilbert_size,
    config=CageClassificationConfig(
        amplitude_tolerance=1e-10,
        cancellation_tolerance=1e-9,
        action_tolerance=1e-9,
    ),
)

rich_print(report.to_rich())

## Lindblad dynamics

In [ ]:
problem = build_type1_cage_lindblad_construction(
    model=model,
    build_result=build_result,
    cage_state=state_vector,
    classification_report=report,
    z_value=signature[1],
    monitor_source="reduced_iz_operators",
    reduced_iz_monitor_content="offdiagonal_only",
    reduced_iz_monitor_decomposition="exact_support",
    jump_operator_design="kinetic_outside_monitor_inside",
    recycling_jump_source="local_rdm_block_reset",
    deduplicate_recycling_regions=True,
    kinetic_jump_grouping="support_greedy",
    max_kinetic_jump_support_size=8,
    check_liouvillian=True,
)

rich_print(problem.to_rich())

In [ ]:
diagnostics = problem.diagnose_dark_subspace(
    hamiltonian=build_result.hamiltonian,
    check_liouvillian_spectrum=False,
    liouvillian_spectrum_method="sparse",
    sparse_liouvillian_eigenvalue_count=4,
)

rich_print(diagnostics.to_rich())

In [ ]:
diagnostics = problem.diagnose_absorbing_projector_symmetry(
    hamiltonian=build_result.hamiltonian,
)

rich_print(diagnostics.to_rich())

In [ ]:
closure = diagnose_monitor_kernel_closure(
    hamiltonian=build_result.hamiltonian,
    monitors=[component.monitor for component in problem.monitor_components],
    target_state=problem.cage_state,
    closure_order=1,
    tolerance=1e-10,
)

rich_print(closure.to_rich())

In [ ]:
readouts = problem.recycler_readouts()

for readout in problem.recycler_readouts():
    print("recycler", readout.recycler_index)
    print("component:", readout.component_index)
    print("variables:", readout.variable_indices)
    print("alpha support patterns:", readout.alpha_support_patterns)
    print("beta support patterns:", readout.beta_support_patterns)
    print("inflow:", readout.inflow_norm)
    print("matrix units:")
    for term in readout.matrix_unit_terms:
        print(" ", term.coefficient, "|", term.target_pattern, "><", term.source_pattern, "|")

In [ ]:
target_energy = np.vdot(state_vector, build_result.hamiltonian @ state_vector)
hamiltonian_residual = np.linalg.norm(
    build_result.hamiltonian @ state_vector - target_energy * state_vector
)

jump_residuals = [
    np.linalg.norm(jump @ state_vector)
    for jump in problem.jumps
]

print("H eigen residual:", hamiltonian_residual)
print("max jump residual:", max(jump_residuals))

### 1. Exact method

In [ ]:
gamma = 1.0
eta = 0.0
times = np.linspace(0.0, 60.0, 21)

density_matrix_initial = (
    eta * pure_density_matrix(state_vector)
    + (1 - eta) * initial_density_matrix(
        build_result.hamiltonian.shape[0],
        kind="mixed",
        rng=0,
    )
)

evolution = problem.evolve(
    hamiltonian=build_result.hamiltonian,
    density_matrix_initial=density_matrix_initial,
    times=times,
    options=LindbladEvolutionOptions(
        method="auto",
        backend="scipy",
        rk4_step_policy="adaptive",
    ),
)

diagnostics = analyze_lindblad_evolution(
    evolution.density_matrices,
    target_state=problem.cage_state,
    hamiltonian=build_result.hamiltonian,
    jumps=problem.jumps,
)

print(diagnostics.fidelities[-1])
print(diagnostics.purities[-1])
print(diagnostics.lindblad_residuals[-1])

plt.plot(times, diagnostics.fidelities, linestyle="--", marker=".", label="fidelity")
plt.plot(times, diagnostics.purities, linestyle="--", marker=".", label="purity")
plt.xlabel("time")
plt.legend()
plt.grid()

### 2. Stochastic Schrodinger method

In [ ]:
times = np.linspace(0.0, 10.0, 51)
psi0 = random_pure_state(basis.n_states, rng=0)

ensemble_result = sample_lindblad_mcwf(
    hamiltonian=hamiltonian_matrix,
    jumps=list(problem.jumps),
    state_initial=psi0,
    times=times,
    rng=0,
    options=McwfOptions(
        n_trajectories=4096,
        store_density_matrices=False,
        store_state_snapshots=False,
        fidelity_targets={"target": state_vector},
        adaptive_time_step=True,
        adaptive_trajectory_block_size=16,
        trajectory_chunk_size=512,
        trajectory_chunk_workers=4,
        max_jump_probability=0.1,
    ),
)

diagnostics = analyze_lindblad_evolution(
    ensemble_result=ensemble_result,
    # target_state=problem.cage_state,
    # density_check_mode="low_rank",
)

print(diagnostics.fidelities[-1])
print(diagnostics.purities[-1])

plt.plot(times, diagnostics.fidelities, linestyle="--", marker=".", label="fidelity")
plt.plot(times, diagnostics.purities, linestyle="--", marker=".", label="purity")
plt.xlabel("time")
plt.legend()
plt.grid()

## Visualization

### 1. Stochastic Schrodinger animation of a single trajector

In [ ]:
times = np.linspace(0.0, 20.0, 201)
psi0 = random_pure_state(basis.n_states, rng=0)

trajectory = run_quantum_jump_trajectory(
    hamiltonian=hamiltonian_matrix,
    jumps=list(problem.jumps),
    state_initial=psi0,
    times=times,
    rng=0,
    max_jump_probability=0.005,
    store_states=True,
    normalize_each_step=True,
    adaptive_time_step=True,
    return_backend_arrays=True,
)

In [ ]:
import matplotlib


matplotlib.use("TkAgg")  # Force interactive GUI window

states = np.asarray(trajectory.states, dtype=np.complex128)

visualizer = StochasticSchrodingerGraphVisualizer.from_trajectory(
    times=trajectory.times,
    states=states,
    hamiltonian=hamiltonian_matrix,
    jump_operators=list(problem.jumps),
    basis_labels=[f"|{index}>" for index in range(basis.n_states)],
    style=HamiltonianGraphStyle(
        figure_size=(12.0, 16.0),
        cmap="coolwarm",
        label_vertices=True,
        vertex_size=18.0,
        colorbar=False,
        edge_colorbar=False,
    ),
)

anim = visualizer.animate(
    layout="kk",
    color_by="amplitude_real",
    edge_color_by="weight_complex",
    interval=300,
    repeat=True,
)

plt.show()
# display(HTML(anim.to_jshtml()))

In [ ]:
from matplotlib.animation import FFMpegWriter

# Save as MP4 using FFMpegWriter
writer = FFMpegWriter(fps=24, metadata=dict(artist='Me'), bitrate=1800)
anim.save("/Users/tandaolin/Downloads/animation.mp4", writer=writer)

### 2. Liouvillian graph

[!WARNING] This is expensive !!

In [ ]:
lio_vis = LiouvillianGraphVisualizer.from_liouvillian(
    problem.build_liouvillian(
        hamiltonian_matrix,
        sparse_format="csr"
    ),
    hilbert_dim=basis.n_states,
    density_matrix=pure_density_matrix(state_vector),
    style=HamiltonianGraphStyle(
        label_vertices=True,
        vertex_size=8.0,
        edge_width=2.0,
        colorbar=True,
        edge_colorbar=True,
        cmap="coolwarm",
        figure_size=(24.0, 20.0),
    ),
)

lio_vis.plot(
    backend="networkx",
    layout="kk",
    color_by="state_phase",
    edge_color_by="weight_complex",
    title="Liouvillian graph: |rho_ij| nodes, complex L edges",
)

In [ ]:
lio_vis.save_graph(
    f"../data/liouvillian_graph.gexf",
    layout_backend="networkx",
    layout="kk",
    color_by="state_phase",
    edge_color_by="weight_complex",
)